# 05 — SIR / SEIR Compartmental Model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

## 1. Load country data

In [ ]:
featured = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])

COUNTRY = 'United States'
POPULATION = 330_000_000

cdf = featured[featured.country == COUNTRY].sort_values('date')
I_obs = cdf['new_cases_7d'].fillna(0).values
print(f"Series length: {len(I_obs)} days")

## 2. Fit SIR model

In [ ]:
from src.models.sir_model import SIRModel

sir = SIRModel()
sir.fit(I_obs, N=POPULATION)
print(f"R₀ = {sir.r0:.3f}")
sir.plot(I_obs, country=COUNTRY)

## 3. Fit SEIR model

In [ ]:
from src.models.sir_model import SEIRModel

seir = SEIRModel(sigma=1/5)
seir.fit(I_obs, N=POPULATION)
print(f"SEIR R₀ = {seir.r0:.3f}")

## 4. 60-day forecast

In [ ]:
import matplotlib.pyplot as plt

forecast = sir.predict_from(I_obs[-1], days=60)
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#050a0f')
ax.set_facecolor('#0a1520')
ax.plot(I_obs[-90:], color='#7a9e90', label='Observed (last 90d)')
ax.plot(range(90, 90+60), forecast, color='#00c896',
        linestyle='--', label='SIR 60-day forecast')
ax.set_title(f'{COUNTRY} — 60-day SIR Forecast  R₀={sir.r0:.2f}', color='#d0e8e0')
ax.legend(facecolor='#0d1b2a', edgecolor='#1a3040', labelcolor='#7a9e90')
ax.grid(True, color='#0f1e2e')
plt.tight_layout()
plt.show()

## 5. R₀ map across countries

In [ ]:
results = []
for country in featured['country'].unique()[:20]:
    sub = featured[featured.country == country].sort_values('date')
    I = sub['new_cases_7d'].fillna(0).values
    pop = int(sub['population'].median()) if 'population' in sub else 1_000_000
    if len(I) < 50 or I.max() < 10:
        continue
    try:
        s = SIRModel()
        s.fit(I, N=max(pop, 1_000_000))
        results.append({'country': country, 'R0': s.r0, 'beta': s.beta, 'gamma': s.gamma})
    except:
        pass

import pandas as pd
r0_df = pd.DataFrame(results).sort_values('R0', ascending=False)
print(r0_df.to_string(index=False))